In [1]:
!pip install torchtext==0.4.0

In [2]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding


In [3]:
!pip install transformers

In [4]:
import torch
from torchtext import data

SEED = 1234

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

TEXT = data.Field(tokenize = 'spacy',
                  tokenizer_language = 'en_core_web_sm',
                  include_lengths = True,
                  pad_first=True)
LABEL = data.LabelField(dtype = torch.float)

In [5]:
from torchtext import datasets

train_data, test_data = datasets.IMDB.splits(TEXT, LABEL)

In [6]:
import random
random.seed(SEED)
train_data, valid_data = train_data.split(random_state = random.seed(SEED))

In [7]:
train_1 = 500
inds = range(len(train_data))
inds = random.sample(inds, train_1)
train_data = [train_data[i] for i in inds]

inds = range(len(valid_data))
inds = random.sample(inds, 500)
valid_data = [valid_data[i] for i in inds]

inds = range(len(test_data))
inds = random.sample(inds, 1000)
test_data = [test_data[i] for i in inds]

In [8]:
# p = 0
# n = 0
# for x in test_data:
#     if x.label=="pos":
#         p+=1
#     else:
#         n+=1
# print(p,n)

In [9]:
import pandas as pd

texts, labels = [], []
with open('positive_reviews.txt', 'r', encoding="utf-8") as f:
    for l in f.readlines():
        texts.append(l.replace('\n', ''))
        labels.append(1)
with open('negative_reviews.txt', 'r', encoding="utf-8") as f:
    for l in f.readlines():
        texts.append(l.replace('\n', ''))
        labels.append(0)

dict = {'text': texts, 'label': labels}
df = pd.DataFrame(dict)
df = df.sample(frac=1)

In [10]:
df.head(10)

,text,label
1034,"""The Lost Day"" is a boring and uninspired shor...",0
1470,This indie film was so deep and thought-provok...,0
1487,"I have to give credit to the filmmakers, they ...",0
614,This movie was a fun and charming musical abou...,1
125,I haven't laughed that hard in a movie theater...,1
1105,"I had high hopes for this movie, but it turned...",0
190,Just saw the latest rom-com and I am absolutel...,1
508,Don't be fooled by the pink outfits and chihua...,1
1446,I can't believe they actually made a sequel to...,0
663,The charming and offbeat 'The Lobster' is a on...,1


In [11]:
# df.shape

In [12]:
# df.head(2)

In [13]:
# train_data[0].label

In [14]:
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer

# Setting up the device for GPU usage

from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [15]:
# Defining some key variables that will be used later on in the training
MAX_LEN = 512
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 2
EPOCHS = 10
LEARNING_RATE = 1e-05
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased')

In [16]:
class DB(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, df):
        if df is not None:
            self.len = len(dataframe) + df.shape[0]
        else:
            self.len = len(dataframe)
        self.data = dataframe
        self.data2 = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __getitem__(self, index):
        # assert(index<self.len)
        if index < len(self.data):
            title = self.data[index].text
            target = self.data[index].label
            if target == 'neg':
                target = 0
            else:
                target = 1
        else:
            index = index-len(self.data)
            title = self.data2.iloc[index,0]
            target = self.data2.iloc[index,1]
        
        inputs = self.tokenizer.encode_plus(
            title,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=True,
            truncation=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']

        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'targets': torch.tensor(target, dtype=torch.long)
        } 
    
    def __len__(self):
        return self.len

In [17]:
train_2 = df
note = f"_{train_1}_{type(train_2)==type(df)}"
training_set = DB(train_data, tokenizer, MAX_LEN, train_2)
validating_set = DB(valid_data, tokenizer, MAX_LEN, None)
testing_set = DB(test_data, tokenizer, MAX_LEN, None)

In [18]:
# test_dataset.__getitem__(1200)

In [19]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [20]:
class DistillBERTClass(torch.nn.Module):
    def __init__(self):
        super(DistillBERTClass, self).__init__()
        self.l1 = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.ReLU()(pooler)
        pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        return output

In [21]:
model = DistillBERTClass()
model.to(device)

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_projector.weight', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


DistillBERTClass(
  (l1): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(i

In [22]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 66,954,241 trainable parameters


In [23]:
# Creating the loss function and optimizer
# loss_function = torch.nn.CrossEntropyLoss()
loss_function = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

In [24]:
# def calcuate_accu(big_idx, targets):
#     n_correct = (big_idx==targets).sum().item()
#     return n_correct

In [25]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import numpy as np


In [26]:
def train(epoch):
    tr_loss = 0
    n_correct = 0
    nb_tr_steps = 0
    nb_tr_examples = 0
    model.train()
    all_preds = []
    all_labels = []
    for _,data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = model(ids, mask).squeeze()
        # print(outputs.shape, outputs.dtype, targets.shape, targets.dtype)
        loss = loss_function(outputs, targets)
        tr_loss += loss.item()
        # big_val, big_idx = torch.max(outputs.data, dim=1)
        preds = outputs.data
        rounded_preds = torch.round(torch.sigmoid(preds))
        all_labels.append(targets.cpu().tolist())
        all_preds.append(rounded_preds.cpu().tolist())
        n_correct = sum((rounded_preds == targets)).float()
        # n_correct += calcuate_accu(big_idx, targets)

        # nb_tr_steps += 1
        # nb_tr_examples+=targets.size(0)
        
        # if _%5000==0:
        #     loss_step = tr_loss/nb_tr_steps
        #     accu_step = (n_correct*100)/nb_tr_examples 
        #     print(f"Training Loss per 5000 steps: {loss_step}")
        #     print(f"Training Accuracy per 5000 steps: {accu_step}")

        optimizer.zero_grad()
        loss.backward()
        # # When using GPU
        optimizer.step()

    # print(f'The Total Accuracy for Epoch {epoch}: {(n_correct*100)/nb_tr_examples}')
    # epoch_loss = tr_loss/nb_tr_steps
    # epoch_accu = (n_correct*100)/nb_tr_examples
    # print(f"Training Loss Epoch: {epoch_loss}")
    # print(f"Training Accuracy Epoch: {epoch_accu}")

    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    # print(all_labels,'\n')
    # print(all_preds)

    print(f"epoch {epoch}: acc {accuracy_score(y_true=all_labels, y_pred=all_preds)*100:.2f}%")
    print(f"epoch {epoch}: f1 {f1_score(y_true=all_labels, y_pred=all_preds)*100:.2f}%")

    return 

In [27]:
def valid(model, testing_loader):
    model.eval()
    tr_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for _,data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)

            outputs = model(ids, mask).squeeze()
            # print(outputs.shape, outputs.dtype, targets.shape, targets.dtype)
            loss = loss_function(outputs, targets)
            tr_loss += loss.item()
            # big_val, big_idx = torch.max(outputs.data, dim=1)
            preds = outputs.data
            rounded_preds = torch.round(torch.sigmoid(preds))
            all_labels.append(targets.cpu().tolist())
            all_preds.append(rounded_preds.cpu().tolist())
            n_correct = sum((rounded_preds == targets)).float()

    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)

    print("label:")
    print(all_labels,'\n')
    print("preds:")
    print(all_preds)

    print(f"validation: acc {accuracy_score(y_true=all_labels, y_pred=all_preds)*100:.2f}%")
    print(f"validation: f1 {f1_score(y_true=all_labels, y_pred=all_preds):.2f}")

    return 

In [ ]:
for epoch in range(EPOCHS):
    train(epoch)


/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:2354: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


epoch 0: acc 57.53%
epoch 0: f1 58.77%


/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:2354: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


epoch 1: acc 72.90%
epoch 1: f1 71.58%


/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:2354: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [ ]:
valid(model, testing_loader)

In [ ]:
# Saving the files for re-use
output_model_file = f'./pytorch_distilbert_news{note}.bin'
output_vocab_file = f'./vocab_distilbert_news{note}.bin'

model_to_save = model
torch.save(model_to_save, output_model_file)
tokenizer.save_vocabulary(output_vocab_file)

print('All files saved')
print('This tutorial is completed')

In [ ]:
note